# Chapter 23 — Integration on Manifolds and Stokes' Theorem

Companion notebook to Chapter 23 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Forms are stored as in the notebook of Chapter 22: a $k$-form is a function
from a point to an alternating array. Integrals over manifolds are computed
as the chapter defines them, by pulling the form back with a positive
parametrization and integrating its component: Gauss–Legendre quadrature in
the non-periodic variables and the trapezoidal rule in the periodic ones.

In [1]:
import itertools
import math
import warnings

import numpy as np
import torch
from torch.func import jacrev, vmap

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor


def sgn(perm):
    n_inv = sum(1 for i in range(len(perm)) for j in range(i + 1, len(perm)) if perm[i] > perm[j])
    return -1 if n_inv % 2 else 1


def antisym(t):
    k = t.dim()
    if k < 2:
        return t.clone()
    return sum(sgn(p) * t.permute(*p) for p in itertools.permutations(range(k)))


def wedge(a, b):
    return antisym(torch.tensordot(a, b, dims=0)) / (math.factorial(a.dim()) * math.factorial(b.dim()))


def dx(i, n):
    v = torch.zeros(n)
    v[i] = 1.0
    return v


def d(omega):
    def d_omega(x):
        J = jacrev(omega)(x)
        k = J.dim() - 1
        Tn = J.movedim(-1, 0)
        return sum((-1) ** j * Tn.movedim(0, j) for j in range(k + 1))
    return d_omega


def pullback(F, omega):
    def pb(x):
        J = jacrev(F)(x)
        w = omega(F(x))
        for _ in range(w.dim()):
            w = torch.tensordot(w, J, dims=([0], [0]))
        return w
    return pb


def interior(v, w):
    return torch.tensordot(v, w, dims=([0], [0]))


def gauss(a, b, n):
    x, w = np.polynomial.legendre.leggauss(n)
    return T(0.5 * (b - a) * x + 0.5 * (a + b)), T(0.5 * (b - a) * w)


def periodic(n, length=2 * math.pi):
    return torch.arange(n) * (length / n), torch.full((n,), length / n)


def integrate(component, rules):
    # integral of a function of u over a box, with one (nodes, weights) rule per variable
    grids = torch.meshgrid(*[r[0] for r in rules], indexing="ij")
    weights = torch.meshgrid(*[r[1] for r in rules], indexing="ij")
    U = torch.stack([g.reshape(-1) for g in grids], dim=1)
    W = torch.stack([w.reshape(-1) for w in weights], dim=1).prod(dim=1)
    return (vmap(component)(U) * W).sum()


def top(form, param):
    # the component of the pull-back of a k-form by a parametrization of a k-dimensional piece
    def comp(u):
        w = pullback(param, form)(u)
        return w[tuple(range(w.dim()))]
    return comp


e2 = [dx(0, 2), dx(1, 2)]
e3 = [dx(i, 3) for i in range(3)]
mu3 = wedge(wedge(e3[0], e3[1]), e3[2])

2.15.0+cu132


## 1. Orientation

Exercise 23.1: the two stereographic charts of the sphere have the
transition map $u\mapsto u/\|u\|^2$, of determinant $-1/\|u\|^4$. They do
not form an oriented atlas; after one of them is composed with a reflection
they do.

In [2]:
phi_N = lambda x: x[:2] / (1 - x[2])
phi_S = lambda x: x[:2] / (1 + x[2])
inv_N = lambda u: torch.cat([2 * u, ((u ** 2).sum() - 1).reshape(1)]) / ((u ** 2).sum() + 1)

u = T([0.7, -1.2])
xs = inv_N(u)
assert torch.allclose(xs.norm(), T(1.0)) and torch.allclose(phi_N(xs), u)
tau = lambda v: phi_S(inv_N(v))
assert torch.allclose(tau(u), u / (u ** 2).sum())
det_tau = torch.linalg.det(jacrev(tau)(u))
assert torch.allclose(det_tau, -1 / (u ** 2).sum() ** 2)
flip = lambda v: v * T([1.0, -1.0])
det_fixed = torch.linalg.det(jacrev(lambda v: flip(tau(v)))(u))
assert det_fixed > 0
print(f"det of the transition map: {det_tau.item():.6f} = -1/|u|^4;  after a reflection: {det_fixed.item():.6f}")

det of the transition map: -0.268464 = -1/|u|^4;  after a reflection: 0.268464


## 2. Densities and the Riemannian volume

The volume density is $\sqrt{\det G}$, with $G = J^\top J$ the Gram matrix
of a parametrization. The area of the sphere computed in two different
charts is the same (Theorem 23.6); the torus of Exercise 23.4.

In [3]:
def volume_density(param):
    def rho(u):
        J = jacrev(param)(u)
        return torch.sqrt(torch.linalg.det(J.T @ J))
    return rho


sph = lambda u: torch.stack([torch.sin(u[0]) * torch.cos(u[1]), torch.sin(u[0]) * torch.sin(u[1]), torch.cos(u[0])])
area_spherical = integrate(volume_density(sph), [gauss(0, math.pi, 40), periodic(64)])
assert abs(area_spherical - 4 * math.pi) < 1e-10

# the stereographic chart, in polar coordinates of the plane: u = rho (cos a, sin a), rho = tan(s), s in (0, pi/2)
stereo = lambda v: inv_N(torch.tan(v[0]) * torch.stack([torch.cos(v[1]), torch.sin(v[1])]))
area_stereo = integrate(volume_density(stereo), [gauss(0, math.pi / 2, 60), periodic(64)])
assert abs(area_stereo - 4 * math.pi) < 1e-8
print(f"area of the sphere: {area_spherical.item():.10f} (spherical chart), {area_stereo.item():.10f} (stereographic chart)")

# the same function integrated against the volume in both charts
h = lambda x: x[2] ** 2 + x[0]
I1 = integrate(lambda v: h(sph(v)) * volume_density(sph)(v), [gauss(0, math.pi, 40), periodic(64)])
I2 = integrate(lambda v: h(stereo(v)) * volume_density(stereo)(v), [gauss(0, math.pi / 2, 60), periodic(64)])
assert abs(I1 - 4 * math.pi / 3) < 1e-10 and abs(I2 - I1) < 1e-8
# without the factor sqrt(det G) the result depends on the chart
J1 = integrate(lambda v: h(sph(v)), [gauss(0, math.pi, 40), periodic(64)])
J2 = integrate(lambda v: h(stereo(v)), [gauss(0, math.pi / 2, 60), periodic(64)])
assert abs(J1 - J2) > 1.0
print(f"integral of z^2 + x: {I1.item():.8f} in both charts;  without sqrt(det G): {J1.item():.4f} and {J2.item():.4f}")

# Exercise 23.4: torus
R, r = 2.0, 0.7
torus = lambda v: torch.stack([(R + r * torch.cos(v[0])) * torch.cos(v[1]), (R + r * torch.cos(v[0])) * torch.sin(v[1]),
                               r * torch.sin(v[0])])
v0 = T([0.4, 1.1])
Jt = jacrev(torus)(v0)
assert torch.allclose(Jt.T @ Jt, torch.diag(T([r ** 2, (R + r * math.cos(0.4)) ** 2])))
area_torus = integrate(volume_density(torus), [periodic(64), periodic(64)])
z2 = integrate(lambda v: torus(v)[2] ** 2 * volume_density(torus)(v), [periodic(64), periodic(64)])
assert abs(area_torus - 4 * math.pi ** 2 * R * r) < 1e-10 and abs(z2 - 2 * math.pi ** 2 * R * r ** 3) < 1e-10
print(f"torus: area {area_torus.item():.8f} = 4 pi^2 R r,  integral of z^2 = {z2.item():.8f} = 2 pi^2 R r^3")

area of the sphere: 12.5663706144 (spherical chart), 12.5663706144 (stereographic chart)
integral of z^2 + x: 4.18879020 in both charts;  without sqrt(det G): 9.8696 and 4.9348
torus: area 55.26978465 = 4 pi^2 R r,  integral of z^2 = 13.54109724 = 2 pi^2 R r^3


## 3. The integral of a form

Example 23.19: $\sigma = i_E\,\mu$ on the unit sphere, with the positive
parametrization by $(\theta, \phi)$; the other orientation changes the
sign. Exercise 23.2.

In [4]:
sigma = lambda x: interior(x, mu3)
comp = top(sigma, sph)
assert torch.allclose(comp(T([0.8, 2.0])), torch.sin(T(0.8)))
I_sigma = integrate(comp, [gauss(0, math.pi, 40), periodic(64)])
assert abs(I_sigma - 4 * math.pi) < 1e-10
swapped = lambda v: sph(v.flip(0))            # the chart (phi, theta) is negative
I_neg = integrate(top(sigma, swapped), [periodic(64), gauss(0, math.pi, 40)])
assert abs(I_neg + 4 * math.pi) < 1e-10
print(f"integral of sigma over the sphere: {I_sigma.item():.10f} = 4 pi;  other orientation: {I_neg.item():.10f}")

# Exercise 23.2: omega = z dx^dy - x dy^dz over the graph z = x^2 + y^2 above the unit disc
omega2 = lambda x: x[2] * wedge(e3[0], e3[1]) - x[0] * wedge(e3[1], e3[2])
graph_xy = lambda q: torch.stack([q[0], q[1], q[0] ** 2 + q[1] ** 2])
q0 = T([0.3, -0.5])
assert torch.allclose(top(omega2, graph_xy)(q0), 3 * q0[0] ** 2 + q0[1] ** 2)
# polar coordinates (t, a) on the disc: a positive chart, since det = t > 0
cap = lambda v: graph_xy(v[0] * torch.stack([torch.cos(v[1]), torch.sin(v[1])]))
I_cap = integrate(top(omega2, cap), [gauss(0, 1, 20), periodic(32)])
assert abs(I_cap - math.pi) < 1e-10
print(f"Exercise 23.2: {I_cap.item():.10f} = pi")

integral of sigma over the sphere: 12.5663706144 = 4 pi;  other orientation: -12.5663706144
Exercise 23.2: 3.1415926536 = pi


## 4. Smooth steps and a partition of unity (Lemmas 23.13 and 23.15)

On the line, with $K = [0, 3]$ and bumps centred at $0, 1, 2, 3$.

In [5]:
def h(t):
    safe = torch.where(t > 0, t, torch.ones_like(t))
    return torch.where(t > 0, torch.exp(-1 / safe), torch.zeros_like(t))


theta = lambda t: h(1 - t) / (h(1 - t) + h(t - 0.5))
beta = lambda t: h(1 - t ** 2)

ts = torch.linspace(-2, 3, 501)
assert torch.all(theta(ts[ts <= 0.5]) == 1) and torch.all(theta(ts[ts >= 1]) == 0)
assert torch.all((theta(ts) >= 0) & (theta(ts) <= 1))
# the derivatives of h tend to 0 at 0: the first four, by automatic differentiation, at t = 0.01
t0 = T(1e-2, requires_grad=True)
val = h(t0)
for k in range(4):
    (val,) = torch.autograd.grad(val, t0, create_graph=True)
    assert 0 < abs(val) < 1e-25
print("the first four derivatives of h at t = 0.01 are below 1e-25")

centres, radius = T([0.0, 1.0, 2.0, 3.0]), 0.8
xs = torch.linspace(-1.5, 4.5, 1201)
bumps = torch.stack([beta((xs - c) / radius) for c in centres])
s = bumps.sum(0)
on_K = (xs >= 0) & (xs <= 3)
c_min = s[on_K].min()
rho = bumps / (s + theta(s / c_min))
assert c_min > 0
assert torch.allclose(rho.sum(0)[on_K], torch.ones(int(on_K.sum())))
assert torch.all(rho >= 0) and torch.all(rho <= 1) and torch.all(rho[:, (xs < -0.8) | (xs > 3.8)] == 0)
print(f"sum of the rho_j on K = [0, 3]: min {rho.sum(0)[on_K].min().item():.15f}, max {rho.sum(0)[on_K].max().item():.15f}")

the first four derivatives of h at t = 0.01 are below 1e-25
sum of the rho_j on K = [0, 3]: min 1.000000000000000, max 1.000000000000000


## 5. Stokes' theorem

Three cases, each computed on both sides.

* A domain of the plane bounded by the curve $r = 1 + 0.3\cos3\phi$, with a
  $1$-form (Green's formula).
* The unit ball of $\mathbb R^3$ with a $2$-form.
* A piece of the paraboloid $z = x^2 + y^2$, a surface with boundary in
  $\mathbb R^3$, with a $1$-form.

In each case the domain is $\{t\leq1\}$ in coordinates $(t,\dots)$, and
the boundary $t = 1$ has the orientation "outward first".

In [6]:
# (a) the plane
radius_of = lambda a: 1 + 0.3 * torch.cos(3 * a)
region = lambda v: v[0] * radius_of(v[1]) * torch.stack([torch.cos(v[1]), torch.sin(v[1])])
one_form = lambda x: torch.stack([torch.sin(x[0] * x[1]) + x[1] ** 3, torch.exp(0.5 * x[0]) * x[1] + x[0] ** 2])
assert torch.linalg.det(jacrev(region)(T([0.5, 1.0]))) > 0           # positive chart
lhs = integrate(top(d(one_form), region), [gauss(0, 1, 30), periodic(200)])
edge = lambda a: region(torch.cat([torch.ones(1), a]))
rhs = integrate(top(one_form, edge), [periodic(200)])
assert abs(lhs - rhs) < 1e-10
print(f"plane:      int_D d omega = {lhs.item():+.10f}   int_dD omega = {rhs.item():+.10f}")

# area by the boundary integral of (x dy - y dx)/2, and the exact area pi (1 + 0.3^2 / 2)
half = lambda x: 0.5 * torch.stack([-x[1], x[0]])
area = integrate(top(half, edge), [periodic(200)])
assert abs(area - math.pi * (1 + 0.045)) < 1e-10
print(f"area from the boundary: {area.item():.10f}")

# (b) the ball
ball = lambda v: v[0] * sph(v[1:])
two_form = lambda x: (x[0] ** 2 * x[2] * wedge(e3[1], e3[2]) + torch.sin(x[0] + x[1]) * wedge(e3[2], e3[0])
                      + x[2] ** 3 * x[1] * wedge(e3[0], e3[1]))
assert torch.linalg.det(jacrev(ball)(T([0.5, 1.0, 2.0]))) > 0
lhs = integrate(top(d(two_form), ball), [gauss(0, 1, 12), gauss(0, math.pi, 24), periodic(32)])
rhs = integrate(top(two_form, sph), [gauss(0, math.pi, 24), periodic(32)])
assert abs(lhs - rhs) < 1e-9
print(f"ball:       int_B d omega = {lhs.item():+.10f}   int_S omega = {rhs.item():+.10f}")

# (c) a surface with boundary in R^3
alpha = lambda x: torch.stack([x[1] * x[2], x[0] ** 2 + x[2], torch.cos(x[0]) * x[1]])
lhs = integrate(top(d(alpha), cap), [gauss(0, 1, 20), periodic(64)])
rim = lambda a: cap(torch.cat([torch.ones(1), a]))
rhs = integrate(top(alpha, rim), [periodic(64)])
assert abs(lhs - rhs) < 1e-10
print(f"paraboloid: int_S d alpha = {lhs.item():+.10f}   int_dS alpha = {rhs.item():+.10f}")

plane:      int_D d omega = -3.0380797049   int_dD omega = -3.0380797049
area from the boundary: 3.2829643230
ball:       int_B d omega = +3.4087053366   int_S omega = +3.4087053366
paraboloid: int_S d alpha = -3.1415926536   int_dS alpha = -3.1415926536


### Consequences

Example 23.19 (three times the volume of the ball), the area of an ellipse
and of a polygon (Exercise 23.3), Stokes on a rectangle (Exercise 23.5),
and the two integrals that count holes (Example 23.20, Exercise 23.6).

In [7]:
# volume of the ball = (1/3) integral of sigma over the sphere
assert abs(I_sigma / 3 - 4 * math.pi / 3) < 1e-10

# ellipse
a_, b_ = 2.0, 0.5
ellipse = lambda t: torch.stack([a_ * torch.cos(t[0]), b_ * torch.sin(t[0])])
assert abs(integrate(top(half, ellipse), [periodic(64)]) - math.pi * a_ * b_) < 1e-12

# polygon: a regular hexagon of circumradius 1, area 3 sqrt(3) / 2, and a non-convex one
def shoelace(P):
    Q = torch.roll(P, -1, dims=0)
    return 0.5 * (P[:, 0] * Q[:, 1] - Q[:, 0] * P[:, 1]).sum()


hexagon = torch.stack([T([math.cos(k * math.pi / 3), math.sin(k * math.pi / 3)]) for k in range(6)])
assert abs(shoelace(hexagon) - 1.5 * math.sqrt(3)) < 1e-12
ell = T([[0.0, 0.0], [2.0, 0.0], [2.0, 1.0], [1.0, 1.0], [1.0, 2.0], [0.0, 2.0]])     # an L of area 3
assert abs(shoelace(ell) - 3.0) < 1e-12
# the segment formula
p1, p2 = T([0.3, -1.0]), T([1.7, 0.4])
seg = lambda t: p1 + t[0] * (p2 - p1)
assert abs(integrate(top(half, seg), [gauss(0, 1, 4)]) - 0.5 * (p1[0] * p2[1] - p2[0] * p1[1])) < 1e-14
print("areas: ellipse pi a b, hexagon 3 sqrt(3)/2, L-shaped polygon 3")

# rectangle [a, b] x [c, d], counter-clockwise
a, b, c, dd = -0.5, 1.2, 0.3, 1.1
rect = lambda v: v
lhs = integrate(top(d(one_form), rect), [gauss(a, b, 20), gauss(c, dd, 20)])
sides = [(T([a, c]), T([b, c])), (T([b, c]), T([b, dd])), (T([b, dd]), T([a, dd])), (T([a, dd]), T([a, c]))]
rhs = sum(integrate(top(one_form, lambda t, A=A, B=B: A + t[0] * (B - A)), [gauss(0, 1, 20)]) for A, B in sides)
assert abs(lhs - rhs) < 1e-12
print(f"rectangle:  int_R d omega = {lhs.item():+.10f}   four sides = {rhs.item():+.10f}")

# winding: the angle form over the boundary of the domain of (a), which contains 0, and of a translate that does not
angle = lambda x: torch.stack([-x[1], x[0]]) / (x ** 2).sum()
w_in = integrate(top(angle, edge), [periodic(400)])
shifted = lambda a: edge(a) + T([2.5, 0.4])
w_out = integrate(top(angle, shifted), [periodic(400)])
assert abs(w_in - 2 * math.pi) < 1e-10 and abs(w_out) < 1e-10
print(f"angle form over the boundary: {w_in.item():.10f} (0 inside), {w_out.item():.2e} (0 outside)")

# Exercise 23.6: sigma / r^3 over an ellipsoid
solid = lambda x: sigma(x) / x.norm() ** 3
ellipsoid = lambda v: sph(v) * T([1.5, 0.8, 1.1])
s_in = integrate(top(solid, ellipsoid), [gauss(0, math.pi, 60), periodic(96)])
s_out = integrate(top(solid, lambda v: ellipsoid(v) + T([0.4, 2.0, 0.3])), [gauss(0, math.pi, 60), periodic(96)])
assert abs(s_in - 4 * math.pi) < 1e-9 and abs(s_out) < 1e-9
vol_ellipsoid = integrate(top(sigma, ellipsoid), [gauss(0, math.pi, 60), periodic(96)]) / 3
assert abs(vol_ellipsoid - 4 * math.pi / 3 * 1.5 * 0.8 * 1.1) < 1e-9
print(f"sigma/r^3 over an ellipsoid: {s_in.item():.10f} = 4 pi (0 inside), {s_out.item():.2e} (0 outside)")

areas: ellipse pi a b, hexagon 3 sqrt(3)/2, L-shaped polygon 3
rectangle:  int_R d omega = -1.0537026419   four sides = -1.0537026419
angle form over the boundary: 6.2831853072 (0 inside), 1.53e-16 (0 outside)
sigma/r^3 over an ellipsoid: 12.5663706144 = 4 pi (0 inside), 3.22e-16 (0 outside)


## 6. Calculus on a graph (Proposition 23.22)

The graph of Figure 23.4 and Exercise 23.9, then a graph with two
components.

In [8]:
def incidence(n, edges):
    D = torch.zeros(len(edges), n)
    for k, (tail, head) in enumerate(edges):
        D[k, tail] = -1.0
        D[k, head] = 1.0
    return D


edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]
D = incidence(4, edges)
n, m, c = 4, 5, 1
assert torch.linalg.matrix_rank(D) == n - c

# (1) the integral of df along a path is the difference of the end values
f = torch.randn(4)
df = D @ f
path = [(0, +1), (2, +1), (4, +1)]          # e1, e3, e5: from node 1 to node 4
assert torch.allclose(sum(sign * df[k] for k, sign in path), f[3] - f[0])

# (4) the dual: flow in minus flow out; its kernel has dimension m - n + c, spanned by the two cycles
y1, y2 = T([1.0, -1.0, 1.0, 0.0, 0.0]), T([0.0, 0.0, 1.0, -1.0, 1.0])
assert torch.allclose(y1 @ D, torch.zeros(4)) and torch.allclose(y2 @ D, torch.zeros(4))
assert m - torch.linalg.matrix_rank(D.T) == m - n + c == 2
y = torch.randn(5)
assert torch.allclose(y @ df, (y @ D) @ f)   # Stokes: integral of df over the chain y = f on the boundary of y
print("rank d = 3, dim ker d^T = 2; the integral of df over a chain is f evaluated on its boundary")

# (3) exactness
alpha, alpha_bad = T([1.0, 2.0, 1.0, 3.0, 2.0]), T([1.0, 2.0, 2.0, 3.0, 2.0])
assert y1 @ alpha == 0 and y2 @ alpha == 0 and y1 @ alpha_bad == 1
pot = T([0.0, 1.0, 2.0, 4.0])
assert torch.allclose(D @ pot, alpha)
residual = lambda al: (al - D @ torch.linalg.lstsq(D, al.unsqueeze(1)).solution.squeeze(1)).norm()
assert residual(alpha) < 1e-12 and residual(alpha_bad) > 0.1
print("alpha = d(0, 1, 2, 4);  alpha' has integral 1 over the first cycle and is not exact")

# the Laplacian
L = D.T @ D
A = torch.zeros(4, 4)
for tail, head in edges:
    A[tail, head] = A[head, tail] = 1.0
assert torch.allclose(L, torch.diag(A.sum(1)) - A)
assert torch.allclose(L, T([[2.0, -1, -1, 0], [-1, 3, -1, -1], [-1, -1, 3, -1], [0, -1, -1, 2]]))
assert torch.allclose(L.sum(1), torch.zeros(4)) and torch.allclose(f @ L @ f, (df ** 2).sum())
flipped = incidence(4, [(1, 0), (0, 2), (2, 1), (1, 3), (3, 2)])
assert torch.allclose(flipped.T @ flipped, L)
print("L = d^T d = degrees - adjacency, independent of the directions of the edges")

# two components: a triangle and a segment
edges2 = [(0, 1), (1, 2), (2, 0), (3, 4)]
D2 = incidence(5, edges2)
assert torch.linalg.matrix_rank(D2) == 5 - 2 and 4 - torch.linalg.matrix_rank(D2.T) == 4 - 5 + 2
eig = torch.linalg.eigvalsh(D2.T @ D2)
assert int((eig.abs() < 1e-10).sum()) == 2
print("two components: rank 3, one independent cycle, eigenvalue 0 of the Laplacian with multiplicity 2")

# one step of gradient descent on <f, L f>/2 averages each value with its neighbours
eta = 0.2
step = f - eta * (L @ f)
assert torch.allclose(step[0], (1 - 2 * eta) * f[0] + eta * (f[1] + f[2]))
assert step @ L @ step < f @ L @ f

rank d = 3, dim ker d^T = 2; the integral of df over a chain is f evaluated on its boundary
alpha = d(0, 1, 2, 4);  alpha' has integral 1 over the first cycle and is not exact
L = d^T d = degrees - adjacency, independent of the directions of the edges
two components: rank 3, one independent cycle, eigenvalue 0 of the Laplacian with multiplicity 2


## 7. Integration by parts: Stein's identity and score matching

Proposition 23.23 for a compactly supported $fX$, by quadrature. Then
Stein's identity for a Gaussian mixture, by Monte Carlo, and the score
matching objective for a Gaussian (Exercise 23.8).

In [9]:
# f X with compact support in the plane: f = bump, X smooth
bump2 = lambda x: h(1 - (x ** 2).sum())
Xf = lambda x: torch.stack([torch.sin(2 * x[1]) + x[0], torch.exp(0.5 * x[0]) * x[1]])
integrand = lambda x: torch.func.grad(bump2)(x) @ Xf(x) + bump2(x) * torch.trace(jacrev(Xf)(x))
disc = lambda v: v[0] * torch.stack([torch.cos(v[1]), torch.sin(v[1])])
total = integrate(lambda v: integrand(disc(v)) * v[0], [gauss(0, 1, 60), periodic(64)])
assert abs(total) < 1e-10
part = integrate(lambda v: bump2(disc(v)) * torch.trace(jacrev(Xf)(disc(v))) * v[0], [gauss(0, 1, 60), periodic(64)])
assert abs(part) > 1e-2
print(f"integral of df(X) + f div X = {total.item():.2e};  the term f div X alone = {part.item():.6f}")

# Stein's identity for a mixture of two Gaussians in R^3
gen = torch.Generator().manual_seed(2)
means = T([[1.0, 0.0, -0.5], [-1.0, 0.5, 0.5]])
log_p = lambda x: torch.logsumexp(-0.5 * ((x - means) ** 2).sum(1), dim=0)       # up to a constant
score = lambda x: torch.func.grad(log_p)(x)
Wx = torch.randn(3, 3, generator=gen)
Xs = lambda x: torch.tanh(Wx @ x) + 0.3 * x
N = 200_000
comp = torch.randint(0, 2, (N,), generator=gen)
samples = means[comp] + torch.randn(N, 3, generator=gen)
stein = vmap(lambda x: score(x) @ Xs(x) + torch.trace(jacrev(Xs)(x)))(samples)
mean, stderr = stein.mean(), stein.std() / math.sqrt(N)
print(f"Stein: mean of <s_p, X> + div X = {mean.item():+.5f} +- {stderr.item():.5f}")
assert abs(mean) < 4 * stderr
divs = vmap(lambda x: torch.trace(jacrev(Xs)(x)))(samples)
assert abs(divs.mean()) > 20 * stderr            # the two terms are far from zero separately

# Exercise 23.8
Asq = torch.randn(3, 3, generator=gen)
Sigma = Asq @ Asq.T + 0.5 * torch.eye(3)
Sinv = torch.linalg.inv(Sigma)
Am = torch.randn(3, 3, generator=gen)
assert torch.allclose(torch.trace(Sinv @ Am @ Sigma), torch.trace(Am))
objective = lambda B: 0.5 * torch.trace(B @ Sigma @ B) - torch.trace(B)
assert torch.allclose(torch.func.grad(objective)(Sinv), torch.zeros(3, 3), atol=1e-12)
for _ in range(5):
    S = torch.randn(3, 3, generator=gen)
    Bm = Sinv + 0.3 * (S + S.T)
    gap = objective(Bm) - objective(Sinv)
    assert torch.allclose(gap, 0.5 * torch.trace((Bm - Sinv) @ Sigma @ (Bm - Sinv))) and gap > 0
print(f"score matching objective of s = -Bx: smallest at B = Sigma^-1, value {objective(Sinv).item():.6f} = -tr(Sigma^-1)/2")

# the divergence estimated with random probes: E[u^T J u] = tr J
x0 = samples[0]
Jx = jacrev(Xs)(x0)
Up = torch.randn(100_000, 3, generator=gen)
est = torch.einsum("ni,ij,nj->n", Up, Jx, Up)
assert abs(est.mean() - torch.trace(Jx)) < 4 * est.std() / math.sqrt(len(Up))
print(f"tr J = {torch.trace(Jx).item():.4f},  mean of u^T J u = {est.mean().item():.4f}")

integral of df(X) + f div X = -6.41e-12;  the term f div X alone = 0.940691
Stein: mean of <s_p, X> + div X = +0.00201 +- 0.00380


score matching objective of s = -Bx: smallest at B = Sigma^-1, value -1.396225 = -tr(Sigma^-1)/2
tr J = 0.9684,  mean of u^T J u = 0.9676


## 8. Probability on the sphere

The direction of a standard Gaussian vector is uniform on the sphere. The
von Mises–Fisher function $q\propto e^{\kappa\langle m, x\rangle}$
integrates to $1$ against the area, with the constant
$\kappa/(4\pi\sinh\kappa)$ on $S^2$; without the factor $\sqrt{\det G}$ the
"density in the chart" does not.

In [10]:
gen = torch.Generator().manual_seed(3)
g = torch.randn(400_000, 3, generator=gen)
dirs = g / g.norm(dim=1, keepdim=True)
test = lambda x: x[..., 2] ** 2 + x[..., 0] * x[..., 1] + x[..., 0]
mc = test(dirs)
exact = integrate(lambda v: test(sph(v)) * volume_density(sph)(v), [gauss(0, math.pi, 40), periodic(64)]) / (4 * math.pi)
assert abs(exact - 1 / 3) < 1e-10
assert abs(mc.mean() - exact) < 4 * mc.std() / math.sqrt(len(mc))
print(f"mean of a test function: {mc.mean().item():.5f} (normalized Gaussians), {exact.item():.5f} (normalized area)")

kappa, m_dir = 2.5, T([0.0, 0.6, 0.8])
q = lambda x: kappa / (4 * math.pi * math.sinh(kappa)) * torch.exp(kappa * (m_dir @ x))
total = integrate(lambda v: q(sph(v)) * volume_density(sph)(v), [gauss(0, math.pi, 60), periodic(96)])
wrong = integrate(lambda v: q(sph(v)), [gauss(0, math.pi, 60), periodic(96)])
assert abs(total - 1) < 1e-10 and abs(wrong - 1) > 0.1
print(f"von Mises-Fisher: integral against the area = {total.item():.10f};  without sqrt(det G): {wrong.item():.4f}")

mean of a test function: 0.33223 (normalized Gaussians), 0.33333 (normalized area)
von Mises-Fisher: integral against the area = 1.0000000000;  without sqrt(det G): 1.7978


## Summary

* Two charts of the sphere with a transition map of negative determinant
  become an oriented atlas after a reflection.
* The integral of a density does not depend on the chart; the integral of a
  function without the factor $\sqrt{\det G}$ does.
* The integral of a form changes sign with the orientation.
* Stokes' theorem holds to rounding error on a curved domain of the plane,
  on the ball and on a surface with boundary in space; the angle form and
  $\sigma/r^3$ count the holes enclosed by a boundary.
* On a graph, the rank of the incidence matrix is $n - c$, its dual has a
  kernel of dimension $m - n + c$ spanned by cycles, and $L = d^\top d$.
* Integration by parts gives Stein's identity and the score matching
  objective.